# Examples of comparing rows in pairs with `compare_pairs`

This notebook compares the wells of one plate of the NF1 dataset (Schwann cells) across their image-based features.
Each row is a well. Two rows are compared when they have a different value in every `different` column and the same value in every `same` column.

In [1]:
import pathlib
import warnings

import pandas as pd

from pairwise_compare import compare_pairs

## Load plate 4

In [2]:
plate4_df = pd.read_parquet(pathlib.Path("../data").resolve(strict=True) / "Plate_4_bulk_feature_selected.parquet")

# wells without an siRNA have a missing value, which would leave them out of the comparisons
plate4_df["Metadata_siRNA"] = plate4_df["Metadata_siRNA"].fillna("No siRNA")

feature_columns = plate4_df.columns[~plate4_df.columns.str.contains("Metadata")].tolist()

print(f"Input dataframe shape: {plate4_df.shape}")

Input dataframe shape: (60, 1182)


## Compare different wells and doses

With no `same` columns, `different` needs at least two columns.
Every pair of wells with a different dose and a different well is compared once.

In [3]:
dose_well_df = compare_pairs(plate4_df, different=["Metadata_Concentration", "Metadata_Well"], features=feature_columns)

print(f"Output dataframe shape: {dose_well_df.shape}")
dose_well_df.head()

Output dataframe shape: (1485, 5)


,pearsons_correlation,Metadata_Concentration__group0,Metadata_Concentration__group1,Metadata_Well__group0,Metadata_Well__group1
0,-0.025999,0.0,0.001,B11,E10
1,-0.035167,0.0,0.001,B11,E4
2,0.148778,0.0,0.001,B11,E7
3,0.030356,0.0,0.001,B11,F10
4,-0.163734,0.0,0.001,B11,F4


The output has one row per pair of wells.
`pearsons_correlation` is the correlation of the two wells across the features, and the other columns hold the values of the two wells of the pair: `group0` is the first well and `group1` the second.

In [4]:
print(f"Output columns:\n{dose_well_df.columns.tolist()}")

Output columns:
['pearsons_correlation', 'Metadata_Concentration__group0', 'Metadata_Concentration__group1', 'Metadata_Well__group0', 'Metadata_Well__group1']


## Compare different siRNAs and wells at the same dose

`same` limits the pairs to wells with the same value in the `same` columns, here the same dose.

In [5]:
sirna_well_df = compare_pairs(
    plate4_df,
    different=["Metadata_siRNA", "Metadata_Well"],
    features=feature_columns,
    same=["Metadata_Concentration"],
)

print(f"Output dataframe shape: {sirna_well_df.shape}")
sirna_well_df.head()

Output dataframe shape: (135, 7)


,pearsons_correlation,Metadata_Concentration__group0,Metadata_Concentration__group1,Metadata_siRNA__group0,Metadata_siRNA__group1,Metadata_Well__group0,Metadata_Well__group1
0,0.128001,0.001,0.001,NF1 Target 1,NF1 Target 2,F10,G10
1,0.190290,0.001,0.001,NF1 Target 1,NF1 Target 2,F10,G4
2,0.173737,0.001,0.001,NF1 Target 1,NF1 Target 2,F10,G7
3,-0.258292,0.001,0.001,NF1 Target 1,Scramble,F10,E10
4,0.089742,0.001,0.001,NF1 Target 1,Scramble,F10,E4


## Leave columns out of the output

`drop` removes columns from the output, while they still decide which wells are compared.
Here the dose and the siRNA are dropped, and only the wells are in the output.

In [6]:
wells_only_df = compare_pairs(
    plate4_df,
    different=["Metadata_siRNA", "Metadata_Well"],
    features=feature_columns,
    same=["Metadata_Concentration"],
    drop=["Metadata_Concentration", "Metadata_siRNA"],
)

print(f"Output columns:\n{wells_only_df.columns.tolist()}")
wells_only_df.head()

Output columns:
['pearsons_correlation', 'Metadata_Well__group0', 'Metadata_Well__group1']


,pearsons_correlation,Metadata_Well__group0,Metadata_Well__group1
0,0.128001,F10,G10
1,0.190290,F10,G4
2,0.173737,F10,G7
3,-0.258292,F10,E10
4,0.089742,F10,E4


## Missing values in the group columns

A well with a missing value in a `same` or `different` column is left out, with a warning.
Without the `fillna` above, the wells without an siRNA would be left out of this comparison.

In [7]:
unfilled_df = pd.read_parquet(pathlib.Path("../data") / "Plate_4_bulk_feature_selected.parquet")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    left_out_df = compare_pairs(unfilled_df, different=["Metadata_siRNA", "Metadata_Well"], features=feature_columns)

filled_df = compare_pairs(plate4_df, different=["Metadata_siRNA", "Metadata_Well"], features=feature_columns)

print(f"Warning: {caught[0].message}")
print(f"Pairs with the missing siRNA left out: {len(left_out_df)}, and with the filled siRNA: {len(filled_df)}")

Pairs with the missing siRNA left out: 675, and with the filled siRNA: 1350
